# Crunchbase Memory Optimization

### Table of Contents
<a id="toc"></a>
1. [Introduction and Setup](#introduction)
2. [Chunk Creation and Exploration](#chunk-exploration)
3. [Chunk Data Type Consistency](#chunk-consistency)
4. [Memory Optimization & SQLite Integration](#conversions-integration)
5. [Analysis and Insights](#analysis)
6. [Conclusion](#conclusion)

### Introduction and Setup
<a id="introduction"></a>
In this project, we demonstrate how to efficiently manage and process a large dataset using **chunking**, data cleaning, and optimization techniques before loading the final result into a SQLite database. We begin by importing the necessary libraries and examining the initial structure of our dataset.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
pd.options.display.max_columns = 99
import sqlite3

In [2]:
first_five = pd.read_csv('crunchbase-investments.csv', nrows=5, encoding='ISO-8859-1')
first_five

,company_permalink,company_name,company_category_code,company_country_code,company_state_code,company_region,company_city,investor_permalink,investor_name,investor_category_code,investor_country_code,investor_state_code,investor_region,investor_city,funding_round_type,funded_at,funded_month,funded_quarter,funded_year,raised_amount_usd
0,/company/advercar,AdverCar,advertising,USA,CA,SF Bay,San Francisco,/company/1-800-flowers-com,1-800-FLOWERS.COM,NaN,USA,NY,New York,New York,series-a,2012-10-30,2012-10,2012-Q4,2012,2000000
1,/company/launchgram,LaunchGram,news,USA,CA,SF Bay,Mountain View,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2012-01-23,2012-01,2012-Q1,2012,20000
2,/company/utap,uTaP,messaging,USA,NaN,United States - Other,NaN,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2012-01-01,2012-01,2012-Q1,2012,20000
3,/company/zoopshop,ZoopShop,software,USA,OH,Columbus,columbus,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,angel,2012-02-15,2012-02,2012-Q1,2012,20000
4,/company/efuneral,eFuneral,web,USA,OH,Cleveland,Cleveland,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2011-09-08,2011-09,2011-Q3,2011,20000


This dataset consists of crowdsourced investment data from **Crunchbase**, a platform dedicated to tracking startup fundraising rounds. The information within this ecosystem—ranging from seed rounds to late-stage acquisitions—is primarily submitted, edited, and maintained by the Crunchbase user community.

While Crunchbase currently operates under a fee-based API model, this specific dataset originates from a period when the data was more openly available online. Because the startup landscape is characterized by constant change, this October 2013 snapshot serves as a historical baseline rather than a real-time reflection of current investments. The raw data for this exploration is hosted on [GitHub](https://github.com/datahoarder/crunchbase-october-2013/blob/master/crunchbase-investments.csv).

### Chunk Creation and Exploration
<a id="chunk-exploration"></a>
Having seen a sample of the dataset, we will now initialize our chunking process. This allows us to inspect the columns and their respective memory footprints more closely without overwhelming system resources.

[Back to Table of Contents](#toc)

In [3]:
chunk_iter = pd.read_csv('crunchbase-investments.csv', chunksize=5000, encoding='ISO-8859-1')

total_missing = None
col_memory_usage = None
column_types = {}

for chunk in chunk_iter:
    # 1. Missing Values
    missing_in_chunk = chunk.isnull().sum()
    if total_missing is None:
        total_missing = missing_in_chunk
    else:
        total_missing += missing_in_chunk
    
    # 2. Memory Usage
    mem_in_chunk = chunk.memory_usage(index=False, deep=True)
    if col_memory_usage is None:
        col_memory_usage = mem_in_chunk
    else:
        col_memory_usage += mem_in_chunk

    # 3. Datatype Tracking
    for col in chunk.columns:
        if col not in column_types:
            column_types[col] = set()
        # Add the string name of the type to our set
        column_types[col].add(str(chunk[col].dtype))

# Final Calculations
total_bytes = col_memory_usage.sum()
total_mb = total_bytes / (1024**2)

# --- RESULTS ---

print("--- Datatypes Found Across All Chunks ---")
for col, types in column_types.items():
    types_str = ", ".join(types)
    warning = " [!] MIXED TYPES" if len(types) > 1 else ""
    print(f"{col}: {types_str}{warning}")

print("\n--- Missing Values Per Column ---")
print(total_missing)

print("\n--- Memory Footprint Per Column (Bytes) ---")
print(col_memory_usage)

print("\n--- Total Memory Footprint ---")
print(f"{total_bytes:,} bytes")
print(f"{total_mb:.2f} MB")

--- Datatypes Found Across All Chunks ---
company_permalink: object
company_name: object
company_category_code: object
company_country_code: object
company_state_code: object
company_region: object
company_city: object
investor_permalink: object
investor_name: object
investor_category_code: object, float64 [!] MIXED TYPES
investor_country_code: object, float64 [!] MIXED TYPES
investor_state_code: object, float64 [!] MIXED TYPES
investor_region: object
investor_city: object, float64 [!] MIXED TYPES
funding_round_type: object
funded_at: object
funded_month: object
funded_quarter: object
funded_year: float64, int64 [!] MIXED TYPES
raised_amount_usd: float64

--- Missing Values Per Column ---
company_permalink             1
company_name                  1
company_category_code       643
company_country_code          1
company_state_code          492
company_region                1
company_city                533
investor_permalink            2
investor_name                 2
investor_categ

The summary above provides a breakdown of datatypes, missing value counts, and memory usage for each column across all chunks. Our initial analysis reveals a total memory footprint of **50.44 MB**. We immediately identified several columns that are either memory-intensive or heavily populated with null values. To streamline our processing, we will drop columns that contain URLs (which offer little analytical value in this context) or those missing more than 90% of their data. Specifically, we are removing `company_permalink`, `investor_permalink`, and `investor_category_code`.

In [4]:
drop_cols = ['investor_permalink', 'company_permalink', 'investor_category_code']
keep_cols = chunk.columns.drop(drop_cols)

In [5]:
keep_cols.tolist

<bound method IndexOpsMixin.tolist of Index(['company_name', 'company_category_code', 'company_country_code',
       'company_state_code', 'company_region', 'company_city', 'investor_name',
       'investor_country_code', 'investor_state_code', 'investor_region',
       'investor_city', 'funding_round_type', 'funded_at', 'funded_month',
       'funded_quarter', 'funded_year', 'raised_amount_usd'],
      dtype='object')>

[Back to Table of Contents](#toc)

### Chunk Data Type Consistency
<a id="chunk-consistency"></a>
Our next objective is to harmonize datatypes across the dataset. A common issue with chunked data processing is **type-shifting**: a column may be interpreted as an integer in one chunk but as a float or object in another due to the presence of missing values. To diagnose these inconsistencies, we developed a custom `profile_chunk` function. We use this function to generate a detailed summary for each chunk, and then leverage those summaries to perform a consistency check. This process allows us to pinpoint exactly which columns change types and identify the specific chunks where these shifts occur.

In [6]:
def profile_chunk(chunk):
    # Memory Footprint (in Megabytes)
    memory_mb = chunk.memory_usage(deep=True).sum() / (1024**2)

    profile_data = []

    for col in chunk.columns:
        # Basic stats
        dtype = chunk[col].dtype
        num_unique = chunk[col].nunique()
        total_rows = len(chunk)
        unique_pct = (num_unique / total_rows) * 100
        
        # Missing values check
        has_missing = chunk[col].isnull().any()
        missing_label = "Yes" if has_missing else "No"
        
        # Get the first 4 values as a comma-separated string
        samples = ", ".join(chunk[col].head(4).astype(str).tolist())
        
        # Integer Candidate check
        is_int_candidate = "No"
        if pd.api.types.is_float_dtype(dtype) and not has_missing:
            if (chunk[col] == chunk[col].apply(int)).all():
                is_int_candidate = "Yes"

        profile_data.append({
            "Column": col,
            "Dtype": dtype,
            "Uniq_Count": num_unique,
            "Uniq_Pct": f"{unique_pct:.2f}%",
            "Has_Missing": missing_label,
            "Int_Candidate": is_int_candidate,
            "Sample_Values": samples
        })

    # Create the summary DataFrame
    summary_df = pd.DataFrame(profile_data)
    
    return summary_df, memory_mb

In [7]:
chunk_iter = pd.read_csv('crunchbase-investments.csv', chunksize=5000, encoding='ISO-8859-1', usecols=keep_cols)

summaries = []

for chunk in chunk_iter:
    summary, footprint = profile_chunk(chunk)
    summaries.append(summary)

In [8]:
dtype_changes = []

# Column/dtype mapping from the first chunk to compare against
baseline_dtypes = summaries[0].set_index('Column')['Dtype']

for i, report in enumerate(summaries):
    current_dtypes = report.set_index('Column')['Dtype']
    
    # Check if any dtypes in the current chunk differ from our baseline
    inconsistent = current_dtypes[current_dtypes != baseline_dtypes]
    
    if not inconsistent.empty:
        for col, dtype in inconsistent.items():
            dtype_changes.append({
                "Chunk": i + 1,
                "Column": col,
                "Original_Dtype": baseline_dtypes[col],
                "New_Dtype": dtype
            })

if not dtype_changes:
    print("Consistency Check: PASS. All columns have consistent types across all 15 chunks.")
else:
    print("Consistency Check: FAIL. Found the following type shifts:")
    display(pd.DataFrame(dtype_changes))

Consistency Check: FAIL. Found the following type shifts:


,Chunk,Column,Original_Dtype,New_Dtype
0,7,funded_year,int64,float64
1,10,investor_country_code,object,float64
2,10,investor_state_code,object,float64
3,10,investor_city,object,float64
4,11,investor_country_code,object,float64
5,11,investor_state_code,object,float64
6,11,investor_city,object,float64


[Back to Table of Contents](#toc)

The results of our consistency check reveal four columns with fluctuating types. By identifying the specific chunks where these changes occur, we can better understand the underlying data quality issues. Below, we examine the first two chunks alongside the problematic chunks highlighted by our check. These summaries will serve as the foundation for our final data cleaning and optimization plan.

In [9]:
summary_exam = [0, 1, 6, 9, 10]
for x in summary_exam:
    print(f"Chunk {x}")
    display(summaries[x])
    print("\n")
    print("-"*85, "\n")

Chunk 0


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,company_name,object,3373,67.46%,No,No,"AdverCar, LaunchGram, uTaP, ZoopShop"
1,company_category_code,object,42,0.84%,Yes,No,"advertising, news, messaging, software"
2,company_country_code,object,1,0.02%,No,No,"USA, USA, USA, USA"
3,company_state_code,object,47,0.94%,Yes,No,"CA, CA, nan, OH"
4,company_region,object,215,4.30%,No,No,"SF Bay, SF Bay, United States - Other, Columbus"
5,company_city,object,542,10.84%,Yes,No,"San Francisco, Mountain View, nan, columbus"
6,investor_name,object,1540,30.80%,No,No,"1-800-FLOWERS.COM, 10Xelerator, 10Xelerator, 1..."
7,investor_country_code,object,45,0.90%,Yes,No,"USA, USA, USA, USA"
8,investor_state_code,object,43,0.86%,Yes,No,"NY, OH, OH, OH"
9,investor_region,object,236,4.72%,No,No,"New York, Columbus, Columbus, Columbus"




------------------------------------------------------------------------------------- 

Chunk 1


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,company_name,object,2946,58.92%,No,No,"Visible World, obopay, Jumptap, Avid Radiophar..."
1,company_category_code,object,40,0.80%,Yes,No,"advertising, mobile, mobile, biotech"
2,company_country_code,object,1,0.02%,No,No,"USA, USA, USA, USA"
3,company_state_code,object,48,0.96%,Yes,No,"NY, CA, MA, PA"
4,company_region,object,229,4.58%,No,No,"New York, SF Bay, Boston, Philadelphia"
5,company_city,object,558,11.16%,Yes,No,"New York, Redwood City, Boston, Philadelphia"
6,investor_name,object,623,12.46%,No,No,"AllianceBernstein, AllianceBernstein, Alliance..."
7,investor_country_code,object,39,0.78%,Yes,No,"USA, USA, USA, USA"
8,investor_state_code,object,38,0.76%,Yes,No,"NY, NY, NY, NY"
9,investor_region,object,146,2.92%,No,No,"New York, New York, New York, New York"




------------------------------------------------------------------------------------- 

Chunk 6


...



------------------------------------------------------------------------------------- 

Chunk 9


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,company_name,object,2045,40.90%,No,No,"Bread, Nuiku, Savvy Cellar Wines, Enigma Techn..."
1,company_category_code,object,42,0.84%,Yes,No,"advertising, software, other, analytics"
2,company_country_code,object,1,0.02%,No,No,"USA, USA, USA, USA"
3,company_state_code,object,41,0.82%,Yes,No,"CA, WA, CA, NY"
4,company_region,object,128,2.56%,No,No,"SF Bay, Seattle, SF Bay, New York"
5,company_city,object,325,6.50%,Yes,No,"San Francisco, Redmond, Redwood City, New York"
6,investor_name,object,2361,47.22%,No,No,"Brendan Wallace, Brent Frei, Brent Harrison, B..."
7,investor_country_code,float64,0,0.00%,Yes,No,"nan, nan, nan, nan"
8,investor_state_code,float64,0,0.00%,Yes,No,"nan, nan, nan, nan"
9,investor_region,object,1,0.02%,No,No,"unknown, unknown, unknown, unknown"




------------------------------------------------------------------------------------- 

Chunk 10


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,company_name,object,1541,53.69%,No,No,"NuORDER, ChaCha, Binfire, Binfire"
1,company_category_code,object,41,1.43%,Yes,No,"fashion, advertising, software, software"
2,company_country_code,object,1,0.03%,No,No,"USA, USA, USA, USA"
3,company_state_code,object,39,1.36%,Yes,No,"CA, IN, FL, FL"
4,company_region,object,106,3.69%,No,No,"Los Angeles, Indianapolis, Bocat Raton, Bocat ..."
5,company_city,object,270,9.41%,Yes,No,"West Hollywood, Carmel, Bocat Raton, Bocat Raton"
6,investor_name,object,1337,46.59%,No,No,"Mortimer Singer, Morton Meyerson, Moshe Ariel,..."
7,investor_country_code,float64,0,0.00%,Yes,No,"nan, nan, nan, nan"
8,investor_state_code,float64,0,0.00%,Yes,No,"nan, nan, nan, nan"
9,investor_region,object,1,0.03%,No,No,"unknown, unknown, unknown, unknown"




------------------------------------------------------------------------------------- 



[Back to Table of Contents](#toc)

### Memory Optimization & SQLite Integration
<a id="conversions-integration"></a>
After analyzing the chunk summaries, we have developed a multi-tiered strategy to clean the data, minimize its memory footprint, and ensure a stable load into SQLite:

* **Schema Consistency:** Resolve data type fluctuations in columns like `investor_city` by explicitly defining them as strings, preventing pandas from misidentifying them as floats due to null values.
* **Low-Cardinality Categorization:** Convert columns with a low number of unique values (such as country codes) to the `category` datatype to drastically reduce memory usage compared to standard strings.
* **Numeric Downcasting:** Optimize storage by converting standard 64-bit integers and floats into more compact 16-bit or 32-bit formats.
* **Pre-load Transformation:** Perform all data cleaning and optimization within the Python "staging" phase, ensuring the data is lean and consistent before reaching the database.
* **SQLite Integration:** Load the optimized data using the `chunk.to_sql` method with the `append` flag to create a durable, queryable storage solution.

In [10]:
# Define explicit dtypes for the initial load
explicit_dtypes = {
    'company_country_code': 'category',
    'funding_round_type': 'category',
    'investor_country_code': 'str',
    'investor_state_code': 'str',
    'investor_city': 'str'
}

chunk_iter = pd.read_csv(
    'crunchbase-investments.csv', 
    chunksize=5000, 
    encoding='ISO-8859-1', 
    usecols=keep_cols,
    dtype=explicit_dtypes,
    parse_dates=['funded_at']
)

# Connect to SQLite
conn = sqlite3.connect('crunchbase.db')

total_memory = 0
for chunk in chunk_iter:
    # --- CLEANUP & OPTIMIZATION STEP ---
    
    # Downcast floats (saves 50% memory for these columns)
    if 'raised_amount_usd' in chunk.columns:
        chunk['raised_amount_usd'] = pd.to_numeric(chunk['raised_amount_usd'], downcast='float')
    
    # Downcast integers (e.g., 2012 doesn't need 64-bit storage)
    if 'funded_year' in chunk.columns:
        chunk['funded_year'] = pd.to_numeric(chunk['funded_year'], downcast='integer')

    # Ensure strings are stripped of whitespace (common CSV cleanup)
    str_cols = chunk.select_dtypes(include=['object']).columns
    chunk[str_cols] = chunk[str_cols].apply(lambda x: x.str.strip())

    # Calculating final memory footprint after changes
    _, chunk_footprint = profile_chunk(chunk)
    total_memory += chunk_footprint

    # --- LOADING STEP ---
    chunk.to_sql('investments', conn, if_exists='append', index=False)


conn.close()
print(f"Final Total Memory: {total_memory}")
print("Cleanup complete. Optimized data loaded to SQLite.")

Final Total Memory: 34.373703956604004
Cleanup complete. Optimized data loaded to SQLite.


In [11]:
conn = sqlite3.connect('crunchbase.db')

# Query a sample of the data
query = "SELECT * FROM investments LIMIT 5;"
sample_df = pd.read_sql(query, conn)

conn.close()

print("--- Database Sample ---")
display(sample_df)

print("\n--- SQLite Column Info ---")
print(sample_df.info())

--- Database Sample ---


,company_name,company_category_code,company_country_code,company_state_code,company_region,company_city,investor_name,investor_country_code,investor_state_code,investor_region,investor_city,funding_round_type,funded_at,funded_month,funded_quarter,funded_year,raised_amount_usd
0,AdverCar,advertising,USA,CA,SF Bay,San Francisco,1-800-FLOWERS.COM,USA,NY,New York,New York,series-a,2012-10-30 00:00:00,2012-10,2012-Q4,2012,2000000.0
1,LaunchGram,news,USA,CA,SF Bay,Mountain View,10Xelerator,USA,OH,Columbus,Columbus,other,2012-01-23 00:00:00,2012-01,2012-Q1,2012,20000.0
2,uTaP,messaging,USA,None,United States - Other,None,10Xelerator,USA,OH,Columbus,Columbus,other,2012-01-01 00:00:00,2012-01,2012-Q1,2012,20000.0
3,ZoopShop,software,USA,OH,Columbus,columbus,10Xelerator,USA,OH,Columbus,Columbus,angel,2012-02-15 00:00:00,2012-02,2012-Q1,2012,20000.0
4,eFuneral,web,USA,OH,Cleveland,Cleveland,10Xelerator,USA,OH,Columbus,Columbus,other,2011-09-08 00:00:00,2011-09,2011-Q3,2011,20000.0



--- SQLite Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   company_name           5 non-null      object 
 1   company_category_code  5 non-null      object 
 2   company_country_code   5 non-null      object 
 3   company_state_code     4 non-null      object 
 4   company_region         5 non-null      object 
 5   company_city           4 non-null      object 
 6   investor_name          5 non-null      object 
 7   investor_country_code  5 non-null      object 
 8   investor_state_code    5 non-null      object 
 9   investor_region        5 non-null      object 
 10  investor_city          5 non-null      object 
 11  funding_round_type     5 non-null      object 
 12  funded_at              5 non-null      object 
 13  funded_month           5 non-null      object 
 14  funded_quarter         5 non-null 

### Analysis and Insights
<a id="analysis"></a>
With the database successfully populated and optimized, we utilized the **pandas and SQLite workflow** to extract key financial insights. By performing aggregations directly within the database, we maintained a low memory profile while answering the following questions:

* **Fund Concentration:** What proportion of the total capital was raised by the top 1% and 10% of startups, and how does this compare to the bottom percentiles?
* **Sector Dominance:** Which company category attracted the highest volume of individual investment rounds?
* **Investor Impact:** Which institutional investor contributed the most capital in total, and which investor provided the highest average funding per startup?
* **Funding Patterns:** Which specific funding rounds (e.g., Series-A, Angel) were the most and least prevalent across the dataset?

In [12]:
conn = sqlite3.connect('crunchbase.db')

# 1. Proportions of total funds raised (Percentiles)
# Pull all non-null raised amounts and sort them to calculate proportions
query_funds = "SELECT raised_amount_usd FROM investments WHERE raised_amount_usd IS NOT NULL ORDER BY raised_amount_usd DESC"
all_funds = pd.read_sql(query_funds, conn)['raised_amount_usd']
total_sum = all_funds.sum()

def get_proportion(series, percentage, top=True):
    count = int(len(series) * (percentage / 100))
    subset = series.head(count) if top else series.tail(count)
    return (subset.sum() / total_sum) * 100

print(f"Top 1% raised: {get_proportion(all_funds, 1):.2f}%")
print(f"Top 10% raised: {get_proportion(all_funds, 10):.2f}%")
print(f"Bottom 10% raised: {get_proportion(all_funds, 10, top=False):.4f}%")
print(f"Bottom 1% raised: {get_proportion(all_funds, 1, top=False):.4f}%")

# 2. Category with the most investments (Count of rounds)
query_cat = """
SELECT company_category_code, COUNT(*) as count 
FROM investments 
GROUP BY company_category_code 
ORDER BY count DESC 
LIMIT 1
"""
print("\nMost popular category:", pd.read_sql(query_cat, conn).iloc[0,0])

# 3. Investor who contributed the most money total
query_top_investor = """
SELECT investor_name, SUM(raised_amount_usd) as total_invested 
FROM investments 
GROUP BY investor_name 
ORDER BY total_invested DESC 
LIMIT 1
"""
print("\nTop total investor:", pd.read_sql(query_top_investor, conn).iloc[0,0])

# 4. Investor who contributed the most money per startup (Average)
query_avg_investor = """
SELECT investor_name, AVG(raised_amount_usd) as avg_investment 
FROM investments 
GROUP BY investor_name 
ORDER BY avg_investment DESC 
LIMIT 1
"""
print("Top investor per startup (Avg):", pd.read_sql(query_avg_investor, conn).iloc[0,0])

# 5. Most and Least popular funding rounds
query_rounds = """
SELECT funding_round_type, COUNT(*) as count 
FROM investments 
GROUP BY funding_round_type 
ORDER BY count DESC
"""
rounds_df = pd.read_sql(query_rounds, conn)
print("\nMost popular round:", rounds_df.iloc[0,0])
print("Least popular round:", rounds_df.iloc[-1,0])

conn.close()

Top 1% raised: 19.29%
Top 10% raised: 49.86%
Bottom 10% raised: 0.2629%
Bottom 1% raised: 0.0013%

Most popular category: software

Top total investor: Kleiner Perkins Caufield & Byers
Top investor per startup (Avg): Marlin Equity Partners

Most popular round: series-a
Least popular round: None


The results of this analysis reveal a classic **Power Law** distribution within the startup ecosystem. With the top 10% of companies capturing nearly half of all funding and the top 1% alone securing almost 20%, investment is heavily concentrated in a small minority of high-growth entities. **Software** remains the most dominant category, largely supported by the high frequency of **Series-A** rounds. Additionally, the presence of major firms like **Kleiner Perkins Caufield & Byers** underscores the impact of institutional capital, while firms like **Marlin Equity Partners** demonstrate a strategy focused on higher average investment per startup.

### Conclusion
<a id="conclusion"></a>
By implementing a strategic ETL pipeline focused on memory efficiency, we successfully transformed a messy, inconsistent dataset into a streamlined SQLite database. Through the use of chunking, explicit type definitions, and numeric downcasting, we reduced the original memory footprint from **50.44 MB** to approximately **34.37 MB**—a significant reduction in resource overhead. This optimization ensured that the data remained highly performant for complex analytical queries, allowing us to uncover critical insights into wealth concentration and investment trends within the Crunchbase ecosystem. The cleanup is complete, the analysis is finalized, and the optimized data is now fully loaded and indexed within SQLite.

[Back to Table of Contents](#toc)